In [4]:
!pip -q install pypdf faiss-cpu langchain_community langchain_text_splitters langchain_openai langchain_huggingface tavily-python ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.2 MB/s eta 0:00:00a 0:00:01


In [5]:
import os
import re
import json
from getpass import getpass
from urllib.parse import urlparse
from typing import TypedDict, List, Dict, Any

import pandas as pd
from IPython.display import display, Markdown

from tavily import TavilyClient
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Set configs

In [6]:
MODEL_NAME = "google/gemini-2.5-flash"
SEARCHES_PER_TOPIC = 4
RESULTS_PER_QUERY = 5
MAX_SOURCES = 6
MAX_CHARS_PER_PAGE = 7000

### Define structured output schemas

In [8]:
class SearchPlan(BaseModel):
    refined_topic: str = Field(description="Cleaner and sharper version of the topic")
    sub_queries: List[str] = Field(description="Search-friendly sub-queries for web research")


class URLSelection(BaseModel):
    selected_ids: List[int] = Field(description="1-based row ids of the best sources to keep")
    

class PageSummary(BaseModel):
    summary: str = Field(description="Short summary of the page")
    top_points: List[str] = Field(description="Most useful facts or claims from the page")
    risks: List[str] = Field(description="Possible risks, limitations, uncertainty, or bias from the page")
    

class SourceItem(BaseModel):
    title: str
    url: str
    why_it_matters: str

class ResearchReport(BaseModel):
    topic: str
    executive_summary: str
    top_findings: List[str]
    risks: List[str]
    sources : List[SourceItem]